<!-- paired-role-banner -->
> 🟧 **실습 노트북 08 · 빈 코드 창**
>
> 오른쪽의 같은 이름 **정답 노트북**과 셀 위치를 맞춰 보세요. 먼저 정답을 가리고 직접
> 작성한 뒤 막힐 때 한 줄씩 비교하는 방식을 권장합니다. `NotImplementedError`가 있는 셀을
> 완성하면 됩니다.

# 08. CNN · RNN · Attention 비교 실습

순차 데이터를 처리하는 세 계열을 같은 관점으로 비교합니다. 먼저 작은 시계열의 다음 값을 예측하고, 이어서 짧은 한국어 문장을 분류합니다.

예상 시간: 100~140분 · 실행 장치: CUDA → MPS → CPU 자동 선택 · 외부 데이터 다운로드 없음

## 한눈에 보는 귀납적 편향

| 모델 | 정보를 섞는 방식 | 강점 | 주의점 |
|---|---|---|---|
| 1D CNN | 가까운 window를 병렬 convolution | 빠름, local pattern | 먼 관계에는 층/팽창 필요 |
| RNN/GRU | 이전 hidden state를 순서대로 갱신 | 순서 표현, streaming | 시간축 병렬화 제한 |
| Self-attention | 모든 token 쌍의 가중합 | 먼 관계, 병렬 학습 | 기본 비용이 $O(T^2)$ |

여기서는 GRU를 RNN 계열의 대표로 사용합니다. 어떤 모델이 항상 우월한 것이 아니라 데이터 크기, sequence 길이, latency 조건에 따라 선택합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,Module.parameters -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from collections import Counter
import math
import re

import pandas as pd
import torch
from torch import nn
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
torch.set_default_device(device)


def parameter_count(model):
    return sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )


print("torch:", torch.__version__, "|", ACCELERATOR.summary())

## Sequence shape 계약

이 노트북은 기본적으로 `[B,T,D]`를 사용합니다.

- `B`: batch size
- `T`: sequence length(time/token)
- `D`: feature/embedding dimension

`Conv1d`만 `[B,C,T]`를 요구하므로 forward 안에서 `transpose(1,2)`합니다. shape 변환을 모델 경계에 모아 두면 디버깅하기 쉽습니다.

## Part A. 작은 시계열: 다음 값 예측

두 주파수가 섞인 noisy sine series를 길이 24의 window로 잘라 다음 한 값을 예측합니다. 미래 window로 validation을 구성해 시간 누수를 막습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.randn,torch.stack -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 1: Part A. 작은 시계열: 다음 값 예측
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("08번 실습의 TODO 1을 완성하세요")

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.mse_loss -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.mse_loss(input, target, reduction=&#x27;mean&#x27;)</code></summary>

#### `F.mse_loss(input, target, reduction='mean')`

- **역할:** 예측과 target의 제곱 오차를 계산합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{MSE}=\frac{1}{N}\sum_{i=1}^{N}(\hat y_i-y_i)^2
$$

- **기호:** $\hat y_i$는 예측, $y_i$는 target입니다.
- **수식 → 코드:** `reduction='mean'`이 모든 원소의 제곱 오차를 평균냅니다.
- **직관:** 큰 오차를 제곱해 더 강하게 벌점하므로 회귀와 reconstruction에 자주 씁니다.
- **shape 확인:** 예측과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.mse_loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# 가장 최근 값을 그대로 예측하는 naive baseline
naive_prediction = valid_x[:, -1, :]
naive_mse = nn.functional.mse_loss(naive_prediction, valid_y).item()
print("naive validation MSE:", naive_mse)

# 시계열 split에서는 train의 마지막 target이 valid의 첫 target보다 과거여야 합니다.
print("first validation input tail/target:", valid_x[0, -3:, 0], valid_y[0, 0])

### 1D CNN

kernel size 3은 한 층에서 인접한 세 시점을 봅니다. 여러 층을 쌓으면 receptive field가 넓어집니다. convolution은 모든 위치를 병렬로 계산할 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.GELU,nn.Linear,Tensor.transpose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.GELU(approximate=&#x27;none&#x27;)</code></summary>

#### `nn.GELU(approximate='none')`

- **역할:** 부드러운 GELU 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>tensor.transpose(dim0, dim1)</code></summary>

#### `tensor.transpose(dim0, dim1)`

- **역할:** 두 축의 위치를 맞바꿉니다.
- **입력·반환:** 두 축 번호를 받고 축이 교환된 view를 반환합니다.
- **이 셀에서 쓰는 이유:** attention 행렬곱 등에서 축 순서를 맞춥니다.
- **주의점:** 반환 텐서는 비연속일 수 있어 이후 `view` 전에 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
Y_{i_{\pi(1)},\ldots,i_{\pi(K)}}=X_{i_1,\ldots,i_K}
$$

- **기호:** $\pi$는 축의 새 순서를 나타내는 permutation입니다.
- **수식 → 코드:** `transpose`는 두 축만 바꾸고 `permute`는 모든 축의 새 순서를 명시합니다.
- **직관:** 원소 값을 바꾸지 않고 각 index가 의미하는 축의 위치만 다시 배치합니다.
- **shape 확인:** 예: `[B,T,H,D] -> [B,H,T,D]`; 결과는 비연속 view일 수 있습니다.
- **공식 문서:** [Tensor.transpose](https://docs.pytorch.org/docs/stable/generated/torch.transpose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 2: 1D CNN
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("08번 실습의 TODO 2을 완성하세요")

### GRU: RNN 계열

GRU는 시점을 순서대로 읽으며 hidden state를 갱신합니다. update/reset gate가 vanilla RNN보다 긴 의존성의 gradient 흐름을 돕습니다. 마지막 hidden state를 다음 값 예측에 사용합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.GRU -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.GRU(input_size, hidden_size, ...)</code></summary>

#### `nn.GRU(input_size, hidden_size, ...)`

- **역할:** gated recurrent unit 층을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}r_t&=\sigma(W_{ir}x_t+b_{ir}+W_{hr}h_{t-1}+b_{hr}),\\z_t&=\sigma(W_{iz}x_t+b_{iz}+W_{hz}h_{t-1}+b_{hz}),\\n_t&=\tanh(W_{in}x_t+b_{in}+r_t\odot(W_{hn}h_{t-1}+b_{hn})),\\h_t&=(1-z_t)\odot n_t+z_t\odot h_{t-1}\end{aligned}
$$

- **기호:** $r_t,z_t$는 reset·update gate, $n_t$는 후보, $h_t$는 hidden state입니다.
- **수식 → 코드:** `nn.GRU`가 time step마다 이 네 식을 계산하고 hidden state를 다음 step으로 넘깁니다.
- **직관:** gate가 과거 정보를 얼마나 지우고 유지할지 학습합니다.
- **shape 확인:** `batch_first=True`이면 입력 `[B,T,D]`, 출력 `[B,T,H]`, 마지막 hidden `[L,B,H]`입니다.
- **공식 문서:** [nn.GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 3: GRU: RNN 계열
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("08번 실습의 TODO 3을 완성하세요")

### Scaled dot-product attention

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V$$

각 query 위치가 key 위치를 얼마나 참고할지 score를 만들고 value의 가중합을 계산합니다. 순서 정보는 연산 자체에 없으므로 positional embedding을 추가합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.sqrt,torch.ones,torch.bool,Tensor.masked_fill,Tensor.softmax,torch.allclose,Tensor.sum,torch.equal,torch.zeros -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 9개 · 수식 5개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.sqrt(input)</code></summary>

#### `torch.sqrt(input)`

- **역할:** 각 원소의 제곱근을 계산합니다.
- **입력·반환:** 0 이상 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표준편차, 거리, diffusion schedule의 scale을 계산합니다.
- **주의점:** 음수 입력은 NaN을 만들며 0 근처 gradient가 매우 커질 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=\sqrt{x_i},\qquad x_i\ge 0
$$

- **기호:** $x_i$는 음수가 아닌 입력입니다.
- **수식 → 코드:** API가 제곱 또는 분산 scale을 원래 단위의 길이로 되돌립니다.
- **직관:** 제곱된 크기를 사람이 해석하기 쉬운 원래 scale로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sqrt](https://docs.pytorch.org/docs/stable/generated/torch.sqrt.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>tensor.masked_fill(mask, value)</code></summary>

#### `tensor.masked_fill(mask, value)`

- **역할:** mask가 참인 위치를 지정 값으로 바꿉니다.
- **입력·반환:** 불리언 mask와 값을 받고 새 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** padding이나 미래 token의 attention score를 선택에서 제외합니다.
- **주의점:** mask는 입력에 broadcast 가능해야 하고 softmax 전 값은 충분히 작아야 합니다.

##### 관련 수식과 코드 연결

$$
s'_{ij}=\begin{cases}s_{ij},&M_{ij}=0\\-\infty,&M_{ij}=1\end{cases}\quad\Rightarrow\quad\operatorname{softmax}(s')_{ij}=0\;\text{if }M_{ij}=1
$$

- **기호:** $s$는 score, $M$은 제외할 위치가 참인 mask입니다.
- **수식 → 코드:** attention에서는 mask 위치를 매우 작은 값으로 채운 뒤 softmax를 적용합니다.
- **직관:** 확률을 나중에 지우는 대신 정규화 분모에도 들어가지 못하게 합니다.
- **shape 확인:** mask는 score에 broadcast 가능해야 하며 보통 `[B, H, T_q, T_k]`를 사용합니다.
- **공식 문서:** [Tensor.masked_fill](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html)

</details>

<details>
<summary><code>tensor.softmax(dim, dtype=None)</code></summary>

#### `tensor.softmax(dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** method 문법으로 attention score나 class logit을 정규화합니다.
- **주의점:** `dim`이 확률 합을 낼 축이며 입력은 일반적으로 raw logit이어야 합니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [Tensor.softmax](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.softmax.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
torch.manual_seed(SEED)
demo_sequence = torch.randn(2, 4, 8)  # [B,T,D]
q = k = v = demo_sequence
scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])  # [B,T,T]

causal_mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
masked_scores = scores.masked_fill(causal_mask, float("-inf"))
attention_weights = masked_scores.softmax(dim=-1)
attended = attention_weights @ v

print("scores/weights:", scores.shape, attention_weights.shape)
print("attended:", attended.shape)
print("first attention matrix:\n", attention_weights[0])
assert torch.allclose(attention_weights.sum(-1), torch.ones(2, 4))
assert torch.equal(attention_weights[:, 0, 1:], torch.zeros(2, 3))

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Parameter,nn.MultiheadAttention,torch.linalg.norm,nn.LayerNorm -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Parameter(data, requires_grad=True)</code></summary>

#### `nn.Parameter(data, requires_grad=True)`

- **역할:** 텐서를 학습 파라미터로 등록합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Parameter](https://docs.pytorch.org/docs/stable/generated/torch.nn.Parameter.html)

</details>

<details>
<summary><code>nn.MultiheadAttention(embed_dim, num_heads, dropout=0.0, batch_first=False, ...)</code></summary>

#### `nn.MultiheadAttention(embed_dim, num_heads, dropout=0.0, batch_first=False, ...)`

- **역할:** 여러 attention head를 병렬 계산하는 모듈을 만듭니다.
- **입력·반환:** embedding·head 설정을 받고 self/cross-attention용 `nn.Module`을 반환합니다.
- **이 셀에서 쓰는 이유:** 표준 MHA를 기준 구현으로 사용하거나 논문 고유 블록과 비교합니다.
- **주의점:** 기본 입력 축, mask shape, 반환 weight 평균 여부를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathrm{head}_h=\operatorname{Attention}(QW_h^Q,KW_h^K,VW_h^V),\qquad \operatorname{MHA}(Q,K,V)=\operatorname{Concat}(\mathrm{head}_1,\ldots,\mathrm{head}_H)W^O
$$

- **기호:** $H$는 head 수이고 $W_h^Q,W_h^K,W_h^V,W^O$는 학습 projection입니다.
- **수식 → 코드:** 모듈이 head 분할, 각 attention, concat, output projection을 묶어 계산합니다.
- **직관:** 여러 표현 부분공간에서 서로 다른 관계를 동시에 학습합니다.
- **shape 확인:** `batch_first=True`이면 주 입력은 `[B,T,E]`, 출력도 `[B,T_q,E]`입니다.
- **공식 문서:** [nn.MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html)

</details>

<details>
<summary><code>torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)</code></summary>

#### `torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)`

- **역할:** 벡터의 크기인 norm을 계산합니다.
- **입력·반환:** 텐서·차수·축을 받고 norm 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 거리, gradient 크기, 정규화 분모를 계산합니다.
- **주의점:** `dim`과 norm 차수에 따라 의미와 반환 shape이 달라집니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [torch.linalg.norm](https://docs.pytorch.org/docs/stable/generated/torch.linalg.vector_norm.html)

</details>

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 4: Scaled dot-product attention
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("08번 실습의 TODO 4을 완성하세요")

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.AdamW,Module.train,Optimizer.zero_grad,Tensor.backward,clip_grad_norm_,Optimizer.step,Tensor.item,Module.eval,torch.inference_mode -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 9개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)</code></summary>

#### `torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)`

- **역할:** weight decay를 gradient update와 분리한 AdamW optimizer를 만듭니다.
- **입력·반환:** 파라미터와 hyperparameter를 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Transformer 계열에서 흔한 최적화 규칙을 재현합니다.
- **주의점:** bias·정규화 파라미터를 decay에서 제외하는 설정도 자주 사용합니다.

##### 관련 수식과 코드 연결

$$
\theta_t=(1-\eta\lambda)\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

- **기호:** $\lambda$는 weight decay이고 나머지 기호는 Adam과 같습니다.
- **수식 → 코드:** 코드의 `weight_decay`가 gradient 기반 Adam update와 분리된 축소 항으로 적용됩니다.
- **직관:** 손실 gradient와 무관하게 큰 weight를 조금씩 줄이는 것이 Adam과의 핵심 차이입니다.
- **shape 확인:** 파라미터와 모멘트 tensor의 shape은 변하지 않습니다.
- **공식 문서:** [torch.optim.AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)</code></summary>

#### `clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)`

- **역할:** 전체 gradient norm이 한계를 넘으면 같은 비율로 축소합니다.
- **입력·반환:** 파라미터와 최대 norm을 받고 clipping 전 전체 norm을 반환합니다.
- **이 셀에서 쓰는 이유:** RNN·Transformer·RL 학습의 폭주하는 update를 제한합니다.
- **주의점:** `backward()` 뒤 `step()` 전에 호출해야 하며 근본 원인을 없애지는 않습니다.

##### 관련 수식과 코드 연결

$$
g' = g\cdot\min\!\left(1,\frac{c}{\lVert g\rVert_p+\epsilon}\right)
$$

- **기호:** $g$는 모든 parameter gradient를 합친 벡터, $c$는 `max_norm`입니다.
- **수식 → 코드:** 전체 norm이 한계를 넘을 때만 모든 gradient를 같은 비율로 줄입니다.
- **직관:** 방향은 유지하면서 update 크기만 제한합니다.
- **shape 확인:** 각 parameter의 gradient shape은 유지됩니다.
- **공식 문서:** [clip_grad_norm_](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def fit_regressor(model, train_x, train_y, epochs=80, learning_rate=0.01):
    torch.manual_seed(SEED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(train_x), train_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if epoch in {1, 20, 40, 60, 80}:
            history.append((epoch, loss.item()))
    model.eval()
    with torch.inference_mode():
        validation_mse = nn.functional.mse_loss(model(valid_x), valid_y).item()
    return history, validation_mse

In [ ]:
torch.manual_seed(SEED)
time_models = {
    "CNN": CNNForecaster(),
    "GRU": GRUForecaster(),
    "Attention": AttentionForecaster(),
}
time_results = []

for name, candidate in time_models.items():
    history, validation_mse = fit_regressor(candidate, train_x, train_y)
    time_results.append(
        {
            "model": name,
            "parameters": parameter_count(candidate),
            "first_train_mse": history[0][1],
            "last_train_mse": history[-1][1],
            "validation_mse": validation_mse,
        }
    )
    print(name, history, "valid=", round(validation_mse, 5))

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
time_comparison = pd.DataFrame(time_results).sort_values("validation_mse")
baseline_row = pd.DataFrame(
    [
        {
            "model": "Naive-last-value",
            "parameters": 0,
            "first_train_mse": float("nan"),
            "last_train_mse": float("nan"),
            "validation_mse": naive_mse,
        }
    ]
)
pd.concat([time_comparison, baseline_row], ignore_index=True).round(6)

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# padding mask와 causal mask는 목적이 다릅니다.
token_mask = torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0]], dtype=torch.bool)
key_padding_mask = ~token_mask  # MultiheadAttention은 무시할 위치가 True
causal_mask = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)

print("token mask [B,T]:\n", token_mask)
print("key padding mask [B,T]:\n", key_padding_mask)
print("causal mask [T,T]:\n", causal_mask)
assert key_padding_mask.shape == (2, 4) and causal_mask.shape == (4, 4)

## Part B. 작은 텍스트 분류

같은 세 계열을 `결제 / 계정 / 배송` 문장에 적용합니다. 데이터가 매우 작으므로 일반화 점수 비교가 아니라 **입력 shape와 모델 연결을 검증하는 overfit smoke test**입니다.

In [ ]:
examples = [
    ("카드가 두 번 결제됐어요", "billing"),
    ("환불 금액이 아직 안 왔어요", "billing"),
    ("청구서 금액을 확인해 주세요", "billing"),
    ("결제 승인이 계속 실패합니다", "billing"),
    ("중복 결제를 취소하고 싶어요", "billing"),
    ("영수증을 다시 보내 주세요", "billing"),
    ("비밀번호 재설정이 안 돼요", "account"),
    ("로그인할 수 없습니다", "account"),
    ("계정이 잠겼어요", "account"),
    ("인증 코드가 오지 않아요", "account"),
    ("이메일 주소를 변경하고 싶어요", "account"),
    ("새 기기 로그인이 차단됐어요", "account"),
    ("택배가 아직 도착하지 않았어요", "delivery"),
    ("배송 조회가 멈췄어요", "delivery"),
    ("주소를 잘못 입력했습니다", "delivery"),
    ("상품이 다른 곳으로 배송됐어요", "delivery"),
    ("배송 예정일을 알려 주세요", "delivery"),
    ("분실된 택배를 찾아 주세요", "delivery"),
]
TEXT_PATTERN = re.compile(r"[가-힣]|[A-Za-z]+|\d+|[^\w\s]")
tokenize = lambda text: TEXT_PATTERN.findall(text.lower())

counts = Counter(token for text, _ in examples for token in tokenize(text))
itos = ["<pad>", "<unk>"] + sorted(counts, key=lambda token: (-counts[token], token))
stoi = {token: index for index, token in enumerate(itos)}
label_names = sorted({label for _, label in examples})
label_to_id = {label: index for index, label in enumerate(label_names)}

print("vocab/classes:", len(itos), label_names)
print("tokens:", tokenize(examples[0][0]))

In [ ]:
encoded_texts = [
    torch.tensor([stoi.get(token, 1) for token in tokenize(text)], dtype=torch.long)
    for text, _ in examples
]
text_ids = nn.utils.rnn.pad_sequence(encoded_texts, batch_first=True, padding_value=0)
text_mask = text_ids.ne(0)
text_labels = torch.tensor([label_to_id[label] for _, label in examples])

print("ids/mask/labels:", text_ids.shape, text_mask.shape, text_labels.shape)
print("first ids:", text_ids[0])
print("first mask:", text_mask[0])
assert text_ids.shape == text_mask.shape

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Embedding,F.relu,torch.arange -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>F.relu(input, inplace=False)</code></summary>

#### `F.relu(input, inplace=False)`

- **역할:** 음수 activation을 0으로 만드는 ReLU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 선형 층 사이에 비선형성을 추가합니다.
- **주의점:** 음수 구간 gradient는 0이며 in-place 연산은 autograd 값을 덮어쓸 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.relu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class CNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, model_dim=24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_dim, padding_idx=0)
        self.convolution = nn.Conv1d(model_dim, 32, kernel_size=3, padding=1)
        self.output = nn.Linear(32, num_classes)

    def forward(self, ids, mask):
        embedded = self.embedding(ids).transpose(1, 2)  # [B,D,T]
        hidden = torch.relu(self.convolution(embedded))
        hidden = hidden.masked_fill(~mask.unsqueeze(1), -1e4)
        pooled = hidden.max(dim=-1).values
        return self.output(pooled)


class GRUTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, model_dim=24):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_dim, padding_idx=0)
        self.gru = nn.GRU(model_dim, 32, batch_first=True)
        self.output = nn.Linear(32, num_classes)

    def forward(self, ids, mask):
        sequence, _ = self.gru(self.embedding(ids))
        lengths = mask.sum(-1).clamp_min(1)
        last = sequence[torch.arange(len(ids)), lengths - 1]
        return self.output(last)

In [ ]:
class AttentionTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, max_length, model_dim=24, num_heads=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_dim, padding_idx=0)
        self.position = nn.Parameter(torch.randn(1, max_length, model_dim) * 0.02)
        self.attention = nn.MultiheadAttention(model_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(model_dim)
        self.output = nn.Linear(model_dim, num_classes)

    def forward(self, ids, mask):
        hidden = self.embedding(ids) + self.position[:, : ids.shape[1]]
        attended, _ = self.attention(
            hidden, hidden, hidden, key_padding_mask=~mask, need_weights=False
        )
        hidden = self.norm(hidden + attended)
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * weights).sum(1) / weights.sum(1).clamp_min(1.0)
        return self.output(pooled)

In [ ]:
torch.manual_seed(SEED)
text_models = {
    "CNN": CNNTextClassifier(len(itos), len(label_names)),
    "GRU": GRUTextClassifier(len(itos), len(label_names)),
    "Attention": AttentionTextClassifier(
        len(itos), len(label_names), max_length=text_ids.shape[1]
    ),
}

architecture_rows = []
for name, candidate in text_models.items():
    logits = candidate(text_ids, text_mask)
    architecture_rows.append(
        {
            "model": name,
            "logits_shape": tuple(logits.shape),
            "parameters": parameter_count(candidate),
        }
    )
    assert logits.shape == (len(examples), len(label_names))
pd.DataFrame(architecture_rows)

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.cross_entropy -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def overfit_text_model(model, steps=100):
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
    losses = []
    for step in range(steps):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(text_ids, text_mask)
        loss = nn.functional.cross_entropy(logits, text_labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step in {0, 24, 49, 99}:
            losses.append((step + 1, loss.item()))
    model.eval()
    with torch.inference_mode():
        predictions = model(text_ids, text_mask).argmax(-1)
    return losses, predictions.eq(text_labels).float().mean().item()


text_results = []
for name, candidate in text_models.items():
    losses, accuracy = overfit_text_model(candidate)
    text_results.append(
        {
            "model": name,
            "parameters": parameter_count(candidate),
            "final_loss": losses[-1][1],
            "train_accuracy": accuracy,
        }
    )
    print(name, losses, "accuracy=", accuracy)

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
comparison = pd.DataFrame(text_results).sort_values("parameters")
print(comparison.round(5))

# 학습 샘플 예측을 확인합니다. 이 점수는 validation 성능이 아닙니다.
selected_model = text_models["Attention"]
with torch.inference_mode():
    probabilities = selected_model(text_ids, text_mask).softmax(-1)
    predicted_ids = probabilities.argmax(-1)

prediction_table = pd.DataFrame(
    {
        "text": [text for text, _ in examples],
        "truth": [label for _, label in examples],
        "prediction": [label_names[index] for index in predicted_ids.tolist()],
        "confidence": probabilities.max(-1).values.tolist(),
    }
)
prediction_table

## 모델 선택 가이드

- 짧은 local motif가 중요하고 latency가 빡빡하면 1D CNN부터 기준 모델을 만듭니다.
- streaming처럼 입력이 순차 도착하고 state를 유지해야 하면 GRU/RNN이 자연스럽습니다.
- 멀리 떨어진 위치의 상호작용과 대규모 병렬 학습이 중요하면 attention이 유리합니다.
- sequence가 매우 길면 vanilla attention의 $T^2$ 메모리를 반드시 측정합니다.
- 실제 선택은 이론만으로 하지 않고 동일 split에서 품질, latency, peak memory, parameter 수를 함께 측정합니다.

이 노트북의 작은 train accuracy는 모델 용량 점검용이며 일반화 성능을 뜻하지 않습니다.

## 따라 치기 TODO + 확장 과제

1. 시계열 window를 `8, 24, 64`로 바꾸고 세 모델의 MSE·학습 시간을 비교하세요.
2. CNN에 dilation을 추가하고 receptive field를 직접 계산하세요.
3. GRU를 `nn.RNN`, `nn.LSTM`으로 바꾸어 parameter 수와 수렴 속도를 비교하세요.
4. attention의 positional embedding을 제거하고 성능 변화를 설명하세요.
5. text 데이터를 group-safe train/validation으로 늘리고 macro F1을 측정하세요.
6. `torch.utils.benchmark`로 sequence length에 따른 forward latency를 측정하세요.
7. attention weight를 반환해 특정 문장이 어느 글자에 집중했는지 표로 만드세요.
8. causal mask가 필요한 생성 모델과 필요 없는 문장 분류 모델의 차이를 예제로 설명하세요.

핵심은 모델 이름이 아니라 **입력/출력 shape, mask 의미, 정보가 섞이는 경로, 계산 비용을 설명할 수 있는가**입니다.